# A/B Test Project — Recommender System

Analysis of the `recommender_system_test` from an international e-commerce store.
The test was run by a previous analyst, who left only the technical specification
and the raw data. The goal here is to **validate whether the test was conducted
correctly** before evaluating its results via a z-test.

## Step 1 — Study objectives

The `recommender_system_test` evaluates the introduction of an **improved
recommendation system** in an international e-commerce store.

**Business hypothesis:** within **14 days of registration**, users in group B
(new funnel) show higher conversion than group A (control) at each stage of the
`product_page → product_cart → purchase` funnel, with a gain of **at least 10%**
at each stage.

**Objective of this analysis:**
1. Validate whether the test was conducted correctly (design, balance, contamination).
2. Evaluate, via z-test, whether group B actually outperforms A in conversions.

**Specification parameters:**
- Audience: 15% of new users from the EU region
- Enrollment: 2020-12-07 to 2020-12-21 · End: 2021-01-01
- Expected participants: ~6000

## Step 2 — Loading and initial inspection

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

marketing    = pd.read_csv('/datasets/ab_project_marketing_events_us.csv')
new_users    = pd.read_csv('/datasets/final_ab_new_users_upd_us.csv')
events       = pd.read_csv('/datasets/final_ab_events_upd_us.csv')
participants = pd.read_csv('/datasets/final_ab_participants_upd_us.csv')

print('marketing   ', marketing.shape)
print('new_users   ', new_users.shape)
print('events      ', events.shape)
print('participants', participants.shape)

In [ ]:
for name, df in [('marketing', marketing), ('new_users', new_users),
                 ('events', events), ('participants', participants)]:
    print(f'===== {name} =====')
    df.info()
    print(df.head(3), '\n')

## Step 3 — Typing, missing values, and duplicates

In [ ]:
# --- Typing: dates → datetime ---
marketing['start_dt']   = pd.to_datetime(marketing['start_dt'])
marketing['finish_dt']  = pd.to_datetime(marketing['finish_dt'])
new_users['first_date'] = pd.to_datetime(new_users['first_date'])
events['event_dt']      = pd.to_datetime(events['event_dt'])

# --- Duplicates (whole rows) ---
for name, df in [('marketing', marketing), ('new_users', new_users),
                 ('events', events), ('participants', participants)]:
    print(f'{name:>14} → duplicates: {df.duplicated().sum()}')

print()

# --- Characterize the nulls in 'details' ---
# Hypothesis: 'details' is only filled in for 'purchase' events.
print('Events WITH details filled in:', events.loc[events['details'].notna(), 'event_name'].unique())
print('\nCount by event type:')
print(events['event_name'].value_counts())

### Step 3 decisions

- **Typing:** `start_dt`, `finish_dt`, `first_date`, and `event_dt` converted to `datetime`.
- **Duplicates:** none found in the four datasets.
- **Missing:** only `details` (in `events`) contains nulls, and this is **structural** —
  the field is only filled in for `purchase` events. **Decision `[autoral]`: keep the nulls**,
  since they represent a legitimate absence of monetary value in navigation events
  (dropping them would discard ~86% of the events).
- **Note for the EDA:** `purchase` (60,314) > `product_cart` (60,120), so the funnel
  **is not strictly sequential** (there are purchases with no recorded cart step).

## Step 4 — Test design validation

In [ ]:
# Which tests exist in the participants table?
print('Tests present:')
print(participants['ab_test'].value_counts(), '\n')

# Isolate our test
rec_sys_test = participants[participants['ab_test'] == 'recommender_system_test'].copy()

# Headcount and group balance
print('recommender_system_test participants:', len(rec_sys_test))
print('\nDistribution by group:')
print(rec_sys_test['group'].value_counts())

In [ ]:
# Users enrolled in MORE THAN ONE test (cross-test contamination)
tests_per_user = participants.groupby('user_id')['ab_test'].nunique()
multi_test = tests_per_user[tests_per_user > 1].index
print('Users in more than one test:', len(multi_test))

# Of those, how many are in OUR test?
contaminated = rec_sys_test[rec_sys_test['user_id'].isin(multi_test)]
print('recommender participants also in another test:', len(contaminated))

# Users appearing in BOTH A and B of our test (serious allocation error)
groups_per_user = rec_sys_test.groupby('user_id')['group'].nunique()
in_two_groups = groups_per_user[groups_per_user > 1]
print('\nrecommender users in A and B simultaneously:', len(in_two_groups))

In [ ]:
# Remove the contaminated users → clean test base
rec_sys_test_clean = rec_sys_test[~rec_sys_test['user_id'].isin(multi_test)].copy()

print('Before cleaning:', len(rec_sys_test))
print('After cleaning:', len(rec_sys_test_clean))
print('\nDistribution by group (clean):')
print(rec_sys_test_clean['group'].value_counts())

### Design validation — findings

- **Headcount:** 3675 participants in `recommender_system_test` (spec expected ~6000 → 39% below).
- **Cross-test contamination:** 887 users (24%) are also in `interface_eu_test`,
  running in the same period and audience (EU). Double exposure prevents attributing the
  effect to the recommender. **Decision `[autoral]`: removed.**
- **Clean base:** 2788 participants (54% below spec).
- **Imbalance:** A=2082 (75%) vs B=706 (25%) — persists after cleaning, so it comes from
  the original allocation, not from contamination. Atypical ratio for an A/B test.
- **Internal allocation:** no user in both A and B simultaneously (ok).

In [ ]:
# Join clean participants with signup data (region + date)
base = rec_sys_test_clean.merge(new_users, on='user_id', how='left')

# (a) REGION — are they all from the EU?
print('Participant regions:')
print(base['region'].value_counts(dropna=False), '\n')

# (b) 14-DAY WINDOW
print('Signup — min:', base['first_date'].min())
print('Signup — max:', base['first_date'].max())
print('Last available event:', events['event_dt'].max())

In [ ]:
# Keep EU only (aligned with the spec)
base = base[base['region'] == 'EU'].copy()

print('Final headcount (EU, no contamination):', len(base))
print('\nDistribution by group:')
print(base['group'].value_counts())

### Sample filtering funnel `[autoral]`

Filtering decisions applied (remove contaminated users, restrict to the EU):

| Step                                     | Participants |
|------------------------------------------|--------------|
| Total in `recommender_system_test`       | 3675         |
| − Contaminated (in 2 tests)              | 2788         |
| − Outside the EU                         | 2594         |
| **Final sample**                         | **2594**     |

Spec expected ~6000 → final sample **57% below**. Group B with only 655 users.
A/B imbalance (~75/25) persists after all cuts → a feature of the original allocation.

In [ ]:
# Actual test window (available events)
test_begin = pd.Timestamp('2020-12-07')
test_end   = events['event_dt'].max()   # 2020-12-30

# Campaigns overlapping the test window
overlap = marketing[
    (marketing['start_dt'] <= test_end) &
    (marketing['finish_dt'] >= test_begin)
]
print('Campaigns active during the test:')
print(overlap[['name', 'regions', 'start_dt', 'finish_dt']])

### Marketing overlap

Two campaigns active in the test window:
- **CIS New Year Gift Lottery** (CIS) — no effect; the sample is EU only.
- **Christmas&New Year Promo** (EU, N.America) — 12/25 to 01/03. **Affects the EU** and covers
  the last ~6 days of the observation window. It hits A and B equally, but may become a
  confounder if the groups have different signup distributions (to be checked in the EDA).

## Step 5 — Exploratory Data Analysis (EDA)

In [ ]:
bg_color, panel_color  = '#0d0221', '#1a0b2e'
text_color, grid_color = '#f2e9ff', '#2d1b4e'
cyan, magenta          = '#01cdfe', '#ff71ce'   # A and B
mint, yellow           = '#05ffa1', '#fffb96'

plt.rcParams.update({
    'figure.facecolor': bg_color, 'axes.facecolor': panel_color,
    'axes.edgecolor': grid_color, 'axes.labelcolor': text_color,
    'axes.titlecolor': text_color, 'text.color': text_color,
    'xtick.color': text_color, 'ytick.color': text_color,
    'grid.color': grid_color, 'figure.dpi': 110,
})

In [ ]:
# Signup distribution per day, split by group
entries = base.groupby([base['first_date'].dt.date, 'group']).size().unstack(fill_value=0)
prop = entries / entries.sum()   # proportion within each group

print(entries, '\n')
print('Signups per day — proportion within each group:')
print(prop.round(3))

In [ ]:
# CHART — Signups by group (temporal confounding)
fig, ax = plt.subplots(figsize=(11, 4.5))
larg, x = 0.4, np.arange(len(prop))
ax.bar(x - larg/2, prop['A'].values, larg, label='A (control)', color=cyan, alpha=0.9)
ax.bar(x + larg/2, prop['B'].values, larg, label='B (recommender)', color=magenta, alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels([d.strftime('%d/%m') for d in prop.index], rotation=45, ha='right', fontsize=8)
ax.set_title('Signups by group (within-group proportion)  ·  temporal confounding',
             fontsize=12, fontweight='bold', pad=12)
ax.set_ylabel("Share of the group's signups")
ax.legend(facecolor=panel_color, edgecolor=grid_color, labelcolor=text_color)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout(); plt.show()

### EDA — temporal distribution of signups

The groups do **not** have an equivalent signup distribution:
- **B concentrates signups early** (12/07 = 19% of B vs 6% of A).
- **A concentrates signups late** (12/21 = 15% of A vs 9% of B).

Implications:
- B had a longer observation window (signed up early; data runs to 12/30).
- A's mass (registered ~12/21) catches the `Christmas&New Year Promo` (starting 12/25)
  within the conversion window; B's mass was already mature when the promo began.
- **Conclusion:** the A vs B comparison is confounded by signup timing. The window and
  promo biases hit the groups unevenly → the z-test result must be read with a strong caveat.

### Decision on the observation window `[autoral]`

The spec asks for conversion "within 14 days of registration", but the usable data ends on
12/29. Anyone who registered near 12/21 never had a full 14 days of observation.

Two alternatives:
- **Option 1** — drop anyone who doesn't complete 14 days. Rigorous, but eliminates a large
  part of an already small sample (655 in group B).
- **Option 2** — accept a **uniform truncated window**, using the events available up to
  12/29, as long as the truncation hits A and B comparably.

**Decision: Option 2**, given the practical infeasibility of Option 1 with this sample — with
the explicit caveat, recorded in the conclusions, that the 14-day window could not be
honored. The comparability check between groups is done above (signup distribution), which
revealed the temporal confounding.

In [ ]:
# Events ONLY from valid participants (EU, no contamination)
base_events = events[events['user_id'].isin(base['user_id'])].merge(
    base[['user_id', 'group']], on='user_id', how='left'
)

# (1) Events per user — evenly distributed between A and B?
ev_per_user = base_events.groupby(['group', 'user_id']).size()
print('Events per user (mean and median):')
print(ev_per_user.groupby('group').agg(['mean', 'median', 'count']), '\n')

# (2) Are users from BOTH samples present in the events?
print('Participants with at least 1 event, by group:')
print(base_events.groupby('group')['user_id'].nunique(), '\n')
print('Total participants by group (base):')
print(base['group'].value_counts(), '\n')

# (3) Events per day
print('Events per day:')
print(base_events.groupby(base_events['event_dt'].dt.date).size())

In [ ]:
# CHART — Events per day
daily_events = base_events.groupby(base_events['event_dt'].dt.date).size()

fig, ax = plt.subplots(figsize=(11, 4.5))
x = range(len(daily_events))
ax.bar(x, daily_events.values, color=mint, alpha=0.85, edgecolor=cyan, linewidth=0.6)
ax.plot(x, daily_events.values, color=cyan, linewidth=1.4, marker='o', markersize=3)
ax.set_xticks(list(x))
ax.set_xticklabels([d.strftime('%d/%m') for d in daily_events.index], rotation=45, ha='right', fontsize=8)
ax.set_title('Events per day  ·  recommender_system_test', fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Number of events'); ax.grid(axis='y', alpha=0.25)
plt.tight_layout(); plt.show()

### EDA — events, presence, and distribution per day

- **Events per user:** A (mean 6.84) is ~20% more active than B (mean 5.65) → another
  sign that the groups are not fully comparable.
- **Presence:** 100% of valid participants generated at least one event in both
  groups (a positive point — no ghost users).
- **Events per day:** peak on 12/21 (end of enrollment), no events on 12/25 (Christmas),
  and a drop to ~zero on 12/30. The useful observation window ends on **12/29**, not 01/01.

In [ ]:
# Unique users who reached each funnel step, by group
steps = ['product_page', 'product_cart', 'purchase']

funnel = {}
for g in ['A', 'B']:
    total_g = base[base['group'] == g]['user_id'].nunique()
    line = {'total_users': total_g}
    for step in steps:
        step_users = base_events[
            (base_events['group'] == g) & (base_events['event_name'] == step)
        ]['user_id'].nunique()
        line[step] = step_users
        line[f'{step}_conv'] = round(step_users / total_g, 4)
    funnel[g] = line

funnel_df = pd.DataFrame(funnel).T
print(funnel_df)

In [ ]:
# CHART — Funnel conversion A vs B
conv_steps = ['product_page_conv', 'product_cart_conv', 'purchase_conv']
conv_a = funnel_df.loc['A', conv_steps].values.astype(float) * 100
conv_b = funnel_df.loc['B', conv_steps].values.astype(float) * 100

fig, ax = plt.subplots(figsize=(9, 5))
larg, x = 0.38, np.arange(len(conv_steps))
b1 = ax.bar(x - larg/2, conv_a, larg, label='A (control)', color=cyan, alpha=0.9)
b2 = ax.bar(x + larg/2, conv_b, larg, label='B (recommender)', color=magenta, alpha=0.9)
ax.set_xticks(x); ax.set_xticklabels(['product_page', 'product_cart', 'purchase'], fontsize=10)
ax.set_ylabel("Conversion (% of the group's participants)")
ax.set_title('Funnel conversion  ·  A vs B', fontsize=13, fontweight='bold', pad=12)
ax.legend(facecolor=panel_color, edgecolor=grid_color, labelcolor=text_color)
ax.grid(axis='y', alpha=0.25); ax.set_ylim(0, 75)
for bars in (b1, b2):
    for b in bars:
        ax.annotate(f'{b.get_height():.1f}%', (b.get_x()+b.get_width()/2, b.get_height()),
                    ha='center', va='bottom', fontsize=8.5)
plt.tight_layout(); plt.show()

### EDA — funnel conversion by group `[autoral]`

**Decision:** conversion computed over each group's total participants
(non-sequential funnel), not as a step-by-step cascade.

| Step           |   A    |   B    | B − A     |
|----------------|--------|--------|-----------|
| product_page   | 65.24% | 56.03% | −9.2 pp   |
| product_cart   | 30.38% | 28.09% | −2.3 pp   |
| purchase       | 31.61% | 29.16% | −2.5 pp   |

**B came in below A at every stage** — the opposite of the hypothesis (B ≥ A + 10%).
In both groups, purchase > product_cart, confirming the non-sequential funnel.

## Step 6 — A/B Test (z-test)

In [ ]:
alpha = 0.05 / 3   # Bonferroni for 3 comparisons

def z_test_prop(successes_a, n_a, successes_b, n_b, step):
    p_a = successes_a / n_a
    p_b = successes_b / n_b
    p_pool = (successes_a + successes_b) / (n_a + n_b)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n_a + 1/n_b))
    z = (p_a - p_b) / se
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    print(f'{step}:')
    print(f'  A={p_a:.4f}  B={p_b:.4f}  z={z:.4f}  p={p_value:.4f}  '
          f'{"REJECT H0" if p_value < alpha else "fail to reject H0"}')
    return p_value

n_a = 1939
n_b = 655
print(f'alpha (Bonferroni) = {alpha:.4f}\n')

z_test_prop(1265, n_a, 367, n_b, 'product_page')
z_test_prop(589,  n_a, 184, n_b, 'product_cart')
z_test_prop(613,  n_a, 191, n_b, 'purchase')

### z-test result

**Multiple-comparison correction `[autoral]`:** three tests → Bonferroni α =
0.05/3 = **0.0167**, fixed before looking at the p-values.

**Hypotheses:** H₀: p_A = p_B  ·  H₁: p_A ≠ p_B (two-tailed)

| Step          | p-value | Decision           | Direction |
|---------------|---------|--------------------|-----------|
| product_page  | 0.0000  | Reject H₀          | A > B     |
| product_cart  | 0.2690  | Fail to reject H₀  | —         |
| purchase      | 0.2404  | Fail to reject H₀  | —         |

The hypothesis (B ≥ A + 10%) was not confirmed at any stage. The only significant
difference (`product_page`) is **in favor of the control**. Cart and purchase show no detectable difference.

## Step 7 — Conclusions

### Nominal test result
Group B (new recommendation system) **did not outperform** group A at any funnel stage.
The only statistically significant difference (`product_page`, p < 0.0001) was
**in favor of the control**. The hypothesis of a ≥10% gain per stage was not confirmed.

### Why this result should NOT be taken as definitive `[autoral]`
The test has serious conduct flaws that compromise the validity of the comparison:

1. **Insufficient sample:** 2594 valid participants vs ~6000 expected (−57%);
   group B with only 655 users → low statistical power.
2. **Structural imbalance:** ~75/25 allocation (A/B), persistent after all
   cuts — atypical and not explained by the specification.
3. **Cross-test contamination:** 24% of the original participants were also in
   `interface_eu_test`, running in the same audience and period (removed, but evidence of
   poor management of concurrent experiments).
4. **Temporal confounding:** B concentrated signups early in the period and A late,
   producing unequal observation windows and different exposure to the Christmas promo.
5. **14-day window not honored:** usable data ends on 12/29; late users
   never had the full 14 days of observation.
6. **Uncontrolled external factor:** `Christmas&New Year Promo` (EU) active in the last
   days of the window, affecting conversion unevenly across groups.

### Recommendation
**Do not implement the new recommendation system based on this test.** The data does not
support the expected gain — and, more importantly, the test is too compromised for a
reliable decision in any direction. It is recommended to **repeat the experiment** with:
balanced allocation, a complete sample (~6000), isolation from other concurrent tests,
a window free of marketing campaigns, and a period that guarantees the full 14 days of
observation for every participant.